In [1]:
from julia.api import Julia
from julia import Main

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import os

import pathlib
from pathlib import Path
import json

#fitters

import pybobyqa
import time
import cma
import csv

Which Fit?

In [2]:
fit_name = "MAP-FD"

Read Files

In [3]:
Main.fit_name = str(fit_name)

TMD_fitting_root = "../"
def include(name):
    path = os.path.join(TMD_fitting_root, name)
    Main.eval(f'include(raw"{path}")')

include(f"Cards/{fit_name}.jl")
include("DY/DY_table_FD.jl")

# Data
data_name = Main.data_name
table_name = Main.table_name

file_root = f"../Data/{data_name}/Cutted/DY"
matrix_root = f"../Data/{data_name}/Covariance_Matrices/DY"
table_root = f"../Tables_JLD2/{table_name}/DY"

initial_params = Main.initial_params

In [4]:
def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].astype("float64")
    return df

df_predictions = dict([])

By file or by experiment?

In [5]:
data_selections = "by_experiment"  # "by_file" or "by_experiment"

In [6]:
experiments =[
    #'ATLAS_7',
    'ATLAS_8', 
    #'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    #'PHENIX',
    'STAR'
]

#["E288","E605","E772","ATLAS", "CMS", "D0", "CDF", "LHCb", "PHENIX", "STAR"]

if data_selections == "by_file":
    file_names = [

    #----------------------------------------------------------------------------
    # ATLAS
    #----------------------------------------------------------------------------

    #"ATLAS/ATLAS_7TeV_y_0_1.csv",
    #"ATLAS/ATLAS_7TeV_y_1_2.csv",
    #"ATLAS/ATLAS_7TeV_y_2_2.4.csv",

    #"ATLAS/ATLAS_8TeV_Q_44_66.csv",
    #"ATLAS/ATLAS_8TeV_Q_116_150.csv",

    #"ATLAS/ATLAS_8TeV_y_0_0.4.csv"
    #"ATLAS/ATLAS_8TeV_y_0.4_0.8.csv"
    #"ATLAS/ATLAS_8TeV_y_0.8_1.2.csv"
    #"ATLAS/ATLAS_8TeV_y_1.2_1.6.csv",
    #"ATLAS/ATLAS_8TeV_y_1.6_2.csv",
    #"ATLAS/ATLAS_8TeV_y_2_2.4.csv",

    #----------------------------------------------------------------------------
    # CDF
    #----------------------------------------------------------------------------

    #"CDF/CDF_RunI.csv",
    #"CDF/CDF_RunII.csv",

    #----------------------------------------------------------------------------
    # CMS
    #----------------------------------------------------------------------------

    #"CMS/CMS_7TeV.csv",
    #"CMS/CMS_8TeV.csv",
    
    #"CMS/CMS_13TeV_y_0_0.4.csv",
    #"CMS/CMS_13TeV_y_0.4_0.8.csv",
    #"CMS/CMS_13TeV_y_0.8_1.2.csv",
    #"CMS/CMS_13TeV_y_1.2_1.6.csv",
    #"CMS/CMS_13TeV_y_1.6_2.4.csv",

    #----------------------------------------------------------------------------
    # D0
    #----------------------------------------------------------------------------

    #"D0/D0_RunI.csv",
    #"D0/D0_RunII.csv",
    #"D0/D0_RunIImu.csv",

    #----------------------------------------------------------------------------
    # LHCb
    #----------------------------------------------------------------------------

    #"LHCb/LHCb_7TeV.csv",
    #"LHCb/LHCb_8TeV.csv",
    #"LHCb/LHCb_13TeV.csv",

    #----------------------------------------------------------------------------
    # Phenix
    #----------------------------------------------------------------------------

    #"PHENIX/PHENIX_200.csv",

    #----------------------------------------------------------------------------
    # STAR
    #----------------------------------------------------------------------------

    #"STAR/STAR_510.csv",

    #----------------------------------------------------------------------------
    # E288
    #----------------------------------------------------------------------------

    #"E288/E288_200_Q_4_5.csv",
    #"E288/E288_200_Q_5_6.csv",
    #"E288/E288_200_Q_6_7.csv",
    #"E288/E288_200_Q_7_8.csv",
    #"E288/E288_200_Q_8_9.csv",
    #"E288/E288_200_Q_10_11.csv",

    #"E288/E288_300_Q_4_5.csv",
    #"E288/E288_300_Q_5_6.csv",
    #"E288/E288_300_Q_6_7.csv",
    #"E288/E288_300_Q_7_8.csv",
    #"E288/E288_300_Q_8_9.csv",
    #"E288/E288_300_Q_10_11.csv",
    #"E288/E288_300_Q_11_12.csv",

    #"E288/E288_400_Q_5_6.csv",
    #"E288/E288_400_Q_6_7.csv",
    #"E288/E288_400_Q_7_8.csv",
    #"E288/E288_400_Q_8_9.csv",
    #"E288/E288_400_Q_10_11.csv",
    #"E288/E288_400_Q_11_12.csv",
    #"E288/E288_400_Q_12_13.csv",
    #"E288/E288_400_Q_13_14.csv",

    #----------------------------------------------------------------------------
    # E605
    #----------------------------------------------------------------------------

    #"E605/E605_Q_7_8.csv",
    #"E605/E605_Q_8_9.csv",
    #"E605/E605_Q_10.5_11.5.csv",
    #"E605/E605_Q_11.5_13.5.csv",
    #"E605/E605_Q_13.5_18.csv",

    #----------------------------------------------------------------------------
    # E772
    #----------------------------------------------------------------------------

    #"E772/E772_Q_5_6.csv",
    #"E772/E772_Q_6_7.csv",
    #"E772/E772_Q_7_8.csv",
    #"E772/E772_Q_8_9.csv",
    #"E772/E772_Q_11_12.csv",
    #"E772/E772_Q_12_13.csv",
    #"E772/E772_Q_13_14.csv",
    #"E772/E772_Q_14_15.csv",
    ]

In [7]:
from pathlib import Path

if data_selections == "by_experiment":
    file_names = []
    for experiment in experiments:
        exp_dir = Path(file_root) / experiment
        for p in sorted(exp_dir.glob("*.csv")):
            file_names.append(str(Path(experiment) / p.name)) 

display(file_names)

['ATLAS_8\\ATLAS8-00y04.csv',
 'ATLAS_8\\ATLAS8-04y08.csv',
 'ATLAS_8\\ATLAS8-08y12.csv',
 'ATLAS_8\\ATLAS8-116Q150.csv',
 'ATLAS_8\\ATLAS8-12y16.csv',
 'ATLAS_8\\ATLAS8-16y20.csv',
 'ATLAS_8\\ATLAS8-20y24.csv',
 'ATLAS_8\\ATLAS8-46Q66.csv',
 'CDF_I\\CDF1.csv',
 'CDF_II\\CDF2.csv',
 'CMS_7\\CMS7.csv',
 'CMS_8\\CMS8.csv',
 'CMS_13\\CMS13-00y04.csv',
 'CMS_13\\CMS13-04y08.csv',
 'CMS_13\\CMS13-08y12.csv',
 'CMS_13\\CMS13-12y16.csv',
 'CMS_13\\CMS13-16y24.csv',
 'D0_I\\D01.csv',
 'D0_II\\D02.csv',
 'D0_II_mu\\D02mu.csv',
 'E288\\E228-200-4Q5.csv',
 'E288\\E228-200-5Q6.csv',
 'E288\\E228-200-6Q7.csv',
 'E288\\E228-200-7Q8.csv',
 'E288\\E228-200-8Q9.csv',
 'E288\\E228-300-11Q12.csv',
 'E288\\E228-300-4Q5.csv',
 'E288\\E228-300-5Q6.csv',
 'E288\\E228-300-6Q7.csv',
 'E288\\E228-300-7Q8.csv',
 'E288\\E228-300-8Q9.csv',
 'E288\\E228-400-11Q12.csv',
 'E288\\E228-400-12Q13.csv',
 'E288\\E228-400-13Q14.csv',
 'E288\\E228-400-5Q6.csv',
 'E288\\E228-400-6Q7.csv',
 'E288\\E228-400-7Q8.csv',
 'E288\\E

Read Data

In [8]:
data_list = dict()
matrix_list = dict()

for file in tqdm(file_names):

    df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
    data_list[file] = df_data
    
    matrix = to_float64(pd.read_csv(f"{matrix_root}/{file}"))
    matrix_list[file] = matrix

data_list_raw = data_list

100%|██████████| 55/55 [00:00<00:00, 286.14it/s]


Prediction

In [9]:
Params = Main.Params_Struct(*[np.float32(x) for x in initial_params]) 
#Main.set_params(Main.VRAM, Params) 

for i in range(10):
    Params = Main.Params_Struct(*[np.float32(x) for x in initial_params])                  
    predictions,t = Main.xsec_dict(Main.rel_paths, Main.VRAM)
    print(round(t*1000,2), "ms")

381.27 ms
84.92 ms
84.78 ms
84.66 ms
85.46 ms
84.62 ms
84.95 ms
84.36 ms
84.33 ms
84.62 ms


In [10]:
def get_file_length():

    file_lengths = dict()

    for file in file_names:

        df = to_float64(pd.read_csv(f"{file_root}/{file}"))

        file_lengths[file] = len(df)

    return file_lengths

file_lengths = get_file_length()

In [11]:
def _norm(p: str) -> str:
    return os.path.normpath(p).replace('\\', '/')

def prediction_reformat(predictions):
    preds = {_norm(k): v for k, v in predictions.items()}  # normalize keys once
    df_predictions = {}

    for file in file_names:
        n = file_lengths[file]
        base = os.path.splitext(file)[0]
        xs = []
        for i in range(n):
            table_path = _norm(os.path.join(table_root, f"{base}/{i}.jld2"))
            xs.append(preds[table_path])
        df_predictions[file] = np.array(xs)

    return df_predictions

df_predictions = prediction_reformat(predictions)


Chi2

In [12]:
ASWZ_b_array = np.linspace(0.12,0.78,12)*5.067731
ASWZ_prediction = np.array([
    -0.08158508158508182,
    -0.1701631701631705,
    -0.2400932400932403,
    -0.34265734265734293,
    -0.37062937062937085,
    -0.4265734265734267,
    -0.4498834498834501,
    -0.44522144522144536,
    -0.4965034965034967,
    -0.5710955710955714,
    -0.6363636363636365,
    -0.7016317016317017
    ])
ASWZ_upper = np.array([
    0.18414918414918402,
    0.11421911421911402,
    0.09557109557109533,
    0.002331002331002141,
    0.016317016317016098,
    -0.034965034965035224,
    -0.034965034965035224,
    -0.011655011655011815,
    -0.034965034965035224,
    -0.05361305361305391,
    -0.05827505827505863,
    -0.04895104895104918
    ])
ASWZ_error = np.array(ASWZ_upper) - np.array(ASWZ_prediction)

def chi2_lattice(): 
    CS_list = []
    for b in ASWZ_b_array :
        Q = 2.0
        CS = Main.CS_total_func(b, Q)
        CS_list.append(CS)
    chi2dN = np.sum( (CS_list - ASWZ_prediction)**2 / ASWZ_error**2 ) / len(ASWZ_b_array)
    return chi2dN

def timed(func):
    t0 = time.perf_counter()
    out = func()
    return out, time.perf_counter() - t0

#chi2dN, t = timed(chi2_lattice)
#print("χ^2/N from LATTICE =", chi2dN, ", took", round(t, 4), "seconds")

In [13]:
def get_chi2dN(df_predictions):

    N_list = dict()
    chi2dN_list = dict()
    chi2_total = 0.0
    N_total = 0

    for file in file_names:

        data_xsec = data_list[file]["xsec"].to_numpy()
        pred_xsec = df_predictions[file]
        diff_xsec = data_xsec - pred_xsec

        covariance_matrix_inv = matrix_list[file].to_numpy()

        N = len(data_xsec)

        chi2 = diff_xsec @ covariance_matrix_inv @ diff_xsec

        chi2_total += chi2
        N_total += N
        chi2dN_list[file] = float(round(chi2/N, 2))
        N_list[file] = N

    chi2dN = chi2_total / N_total
    return chi2dN, chi2dN_list, N_list

chi2dN, chi2dN_list, N_list = get_chi2dN(df_predictions)

print(f"Total χ^2/N = {chi2dN:.2f}")
display(chi2dN_list)

Total χ^2/N = 1.32


{'ATLAS_8\\ATLAS8-00y04.csv': 5.8,
 'ATLAS_8\\ATLAS8-04y08.csv': 4.73,
 'ATLAS_8\\ATLAS8-08y12.csv': 4.0,
 'ATLAS_8\\ATLAS8-116Q150.csv': 0.61,
 'ATLAS_8\\ATLAS8-12y16.csv': 6.72,
 'ATLAS_8\\ATLAS8-16y20.csv': 4.72,
 'ATLAS_8\\ATLAS8-20y24.csv': 2.35,
 'ATLAS_8\\ATLAS8-46Q66.csv': 2.09,
 'CDF_I\\CDF1.csv': 0.59,
 'CDF_II\\CDF2.csv': 1.39,
 'CMS_7\\CMS7.csv': 1.59,
 'CMS_8\\CMS8.csv': 1.59,
 'CMS_13\\CMS13-00y04.csv': 2.13,
 'CMS_13\\CMS13-04y08.csv': 1.41,
 'CMS_13\\CMS13-08y12.csv': 0.36,
 'CMS_13\\CMS13-12y16.csv': 0.21,
 'CMS_13\\CMS13-16y24.csv': 0.25,
 'D0_I\\D01.csv': 0.53,
 'D0_II\\D02.csv': 0.5,
 'D0_II_mu\\D02mu.csv': 0.22,
 'E288\\E228-200-4Q5.csv': 0.25,
 'E288\\E228-200-5Q6.csv': 0.78,
 'E288\\E228-200-6Q7.csv': 0.92,
 'E288\\E228-200-7Q8.csv': 0.99,
 'E288\\E228-200-8Q9.csv': 0.68,
 'E288\\E228-300-11Q12.csv': 0.24,
 'E288\\E228-300-4Q5.csv': 0.83,
 'E288\\E228-300-5Q6.csv': 0.48,
 'E288\\E228-300-6Q7.csv': 0.34,
 'E288\\E228-300-7Q8.csv': 0.23,
 'E288\\E228-300-8Q9.csv': 

Objective

In [14]:
def objective(params):
    try:
        params_cl = Main.Params_Struct(*[np.float32(x) for x in params])
        Main.set_params(Main.VRAM, params_cl)

        predictions, t = Main.xsec_dict(Main.rel_paths, Main.VRAM)

        df_predictions = prediction_reformat(predictions)
        chi2dN, _, _ = get_chi2dN(df_predictions)

        if not np.isfinite(chi2dN): 
            return 1e5
        return chi2dN

    except Exception as e:
        return 1e5

objective(initial_params)

np.float64(1.3243595121630614)

Random

In [15]:
rng = np.random.default_rng()
def generate_gaussian_data():

    data_list = dict()

    for file in file_names:
        df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
        xsecs = df_data["xsec"].to_numpy(np.float64)

        cov_inv = np.asarray(matrix_list[file], np.float64)
        cov_inv = 0.5 * (cov_inv + cov_inv.T)
        cov = np.linalg.inv(cov_inv)
        cov = 0.5 * (cov + cov.T)

        eps = np.finfo(cov.dtype).eps * np.max(np.diag(cov))
        L = np.linalg.cholesky(cov + eps * np.eye(len(xsecs)))

        df_data["xsec"] = xsecs + L @ rng.standard_normal(len(xsecs))
        data_list[file] = df_data

    return data_list

In [16]:
data_list = generate_gaussian_data()
for file in file_names:

    df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
    xsecs = df_data["xsec"].to_numpy()

    cov_mat_inv = matrix_list[file]
    cov_mat = np.linalg.inv(cov_mat_inv)
    cov_diag = np.diag(cov_mat)
    errors = np.sqrt(cov_diag)

    diff = (data_list[file]["xsec"].to_numpy() - xsecs)/errors
    print(f"diff: {diff}")

diff: [0.78030988 0.56229456 0.6494364  0.67154802 0.61370598 0.61342461]
diff: [-1.72751842 -1.64837764 -1.838956   -1.67286628 -1.67447943 -1.78924726]
diff: [0.25420087 0.28267298 0.26485103 0.28207795 0.40022953 0.37151085]
diff: [ 1.56085632  1.15961326  1.08760686  0.85272855  1.31168272  0.99059049
  0.46353606 -0.1255068 ]
diff: [-0.9055724  -0.77287532 -1.07068635 -1.21437301 -1.16397751 -1.08498836]
diff: [-0.13232618 -0.34565245 -0.40282484 -0.14689165 -0.22028813 -0.433709  ]
diff: [-1.36077286 -1.59352108 -1.76111706 -1.62555056 -1.70105631 -1.75820549]
diff: [0.07363706 0.04751994 0.07573698 0.35226676]
diff: [-0.11610285  1.03355996 -0.43717584  0.76105595 -1.10587457  0.9877009
  0.01944988  1.8905964   0.02204898  0.8839636  -0.69155844  0.17370179
 -0.25161424  0.5327438  -0.16462069 -1.46397358  1.49213679  0.99509951
  0.11531084  0.15548112  0.68580058  0.80108484  0.82567472  1.00261041
 -0.22565837]
diff: [-1.75599599  0.26027034 -0.00606122 -0.74232102 -0.409616

In [17]:
objective(initial_params)

np.float64(2.346974239133936)

Bounds

In [18]:
bounds_raw = [
    (0.2, 0.5),   

    (0.0, 0.7),   
    (0.0, 0.1),  
    (0.0, 0.05),   
    (0.0, 0.3),    
    (0.0, 20.0),  

    (0.0, 10.0),   
    (0.0, 10.0),  
    (0.0, 10.0),   
    (0.0, 20.0),    
    (0.0, 3.0),  

    (0, 20.0),    
]

lower_bounds, upper_bounds = np.array(list(zip(*bounds_raw)))

In [19]:
def objective_normalized(params):

    normalized_params = lower_bounds + params * (upper_bounds - lower_bounds)

    return objective(normalized_params)

def objective_log(params):
    return np.log10(objective_normalized(params))

def normalize_params(params):
    return (params - lower_bounds) / (upper_bounds - lower_bounds)

def denormalize_params(params):
    return lower_bounds + params * (upper_bounds - lower_bounds)

theta0 = normalize_params(np.array(initial_params))
dim = len(bounds_raw)

In [24]:
lower_bounds, upper_bounds = np.array(list(zip(*bounds_raw)))

theta0 = normalize_params(np.array(initial_params))
dim = len(bounds_raw)

print(f"initial chi2: {round(objective_normalized(theta0), 3)}")  

res = pybobyqa.solve(
        objective_log, theta0,
        bounds=(np.zeros(dim), np.ones(dim)),
        maxfun=10000,
        rhobeg=0.1,
        rhoend=1e-6,
        scaling_within_bounds=True,
        seek_global_minimum=True, 
    )

initial chi2: 2.293


In [26]:
best_chi2dN = 10**(res.f)
best_chi2dN

np.float64(2.2840961323621642)

Global Minimize

Py-Bobyqa Fit

In [22]:
N_replicas = 200

for i in tqdm(range(N_replicas)):

    data_list = generate_gaussian_data()

    res = pybobyqa.solve(
            objective_log, theta0,
            bounds=(np.zeros(dim), np.ones(dim)),
            maxfun=10000,
            rhobeg=0.1,
            rhoend=1e-6,
            scaling_within_bounds=True,
            seek_global_minimum=True, 
        )
    
    best_chi2dN = 10**(res.f)
    optimal_params_normalized = res.x
    optimal_params = denormalize_params(optimal_params_normalized)

    fmt = lambda x: f"{x:.5g}"     
    with open("replicas.csv", "a", newline="") as f:
        csv.writer(f).writerow([i, fmt(best_chi2dN), *map(fmt, np.asarray(optimal_params))])

    print(f"best chi2: {fmt(best_chi2dN)}")

  0%|          | 0/200 [01:44<?, ?it/s]


KeyboardInterrupt: 

Fit Results

In [ ]:
optimal_params_normalized = res.x
optimal_params = denormalize_params(optimal_params_normalized)
best_chi2dN = 10**(res.f)
N_evals = res.nf

if res.flag == 0:
    print("Optimization successful.")
else:
    print("Optimization failed.")

print()
print("# of Evals =", N_evals)
print()
print("Best χ^2/N =", round(best_chi2dN, 2))
print()
print("Best parameters (normalized):")
print(", ".join(f"{x:.3g}" for x in optimal_params_normalized))
print()
print("Best parameters (unnormalized):")
print(", ".join(f"{x:.3g}" for x in optimal_params))

Optimization successful.

# of Evals = 2821

Best χ^2/N = 2.66

Best parameters (normalized):
0.74, 0.513, 0.505, 0.5, 0.498, 0.501, 0.492, 0.583, 0.71, 0.523, 0.512, 0.665

Best parameters (unnormalized):
0.372, 0.779, 0.284, -0.000584, -0.136, 0.0676, -0.493, 5.01, 12.6, 1.37, 0.739, 9.88


In [ ]:
res.flag

0

In [ ]:
display(initial_params)

array([ 3.72e-01,  7.79e-01,  2.84e-01, -5.84e-04, -1.36e-01,  6.76e-02,
       -4.93e-01,  5.01e+00,  1.26e+01,  1.37e+00,  7.39e-01,  9.88e+00])

In [ ]:
params_cl = Main.Params_Struct(*[np.float32(x) for x in optimal_params])
Main.set_params(Main.VRAM, params_cl)
predictions,t = Main.xsec_dict(Main.rel_paths, Main.VRAM)

df_predictions = prediction_reformat(predictions)
chi2dN, chi2dN_list, N_list = get_chi2dN(df_predictions)

df = pd.DataFrame.from_dict(chi2dN_list, orient="index", columns=["χ^2/N"])
df["N"] = pd.Series(N_list)
df = df.reset_index().rename(columns={"index":"experiment"})
df["experiment"] = df["experiment"].apply(lambda s: Path(s).stem)
df = df[["experiment", "N", "χ^2/N"]]

styled = (
    df.style
      .set_properties(**{"text-align": "center"})           
      .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])  
)
fmt = {col: "{:.2f}" for col in df.select_dtypes(include="float").columns}
display(styled.format(fmt))

total_N = int(df["N"].sum())
print("Total N:", total_N)

,experiment,N,χ^2/N
0,ATLAS7-00y10,6,20.89
1,ATLAS7-10y20,6,11.54
2,ATLAS7-20y24,6,3.05
3,ATLAS8-00y04,6,0.27
4,ATLAS8-04y08,6,0.35
5,ATLAS8-08y12,6,0.24
6,ATLAS8-116Q150,8,0.30
7,ATLAS8-12y16,6,0.25
8,ATLAS8-16y20,6,0.21
9,ATLAS8-20y24,6,0.24


Total N: 460


Plots

In [ ]:
plots_list = dict([])

for file in file_names:

    df = pd.read_csv(f"{file_root}/{file}")

    qT_array = df["qT_mean"]
    data_array = df["xsec"]
    prediction_array = df_predictions[file]
    
    ratio_array = np.array(prediction_array)/df["xsec"]
    matrix = np.linalg.inv(matrix_list[file].to_numpy())
    error_uncor_array = np.sqrt(np.diag(matrix))
    error_uncor_ratio_array = error_uncor_array/data_array

    df_plot = pd.DataFrame([])

    df_plot["qT_array"] = qT_array
    df_plot["data_array"] = data_array
    df_plot["prediction_array"] = prediction_array
    df_plot["ratio_array"] = ratio_array
    df_plot["error_uncor_array"] = error_uncor_array
    df_plot["error_uncor_ratio_array"] = error_uncor_ratio_array

    plots_list[file] = df_plot

def fmt(x):  
    return f"{float(x):g}"

#for file in list(plots_list.keys()):
#    source_df = pd.read_csv(f"{file_root}/{file}")
#
#    q_bucket = source_df["Q_mean"].round(6)
#    if q_bucket.nunique() == 1:
#        continue
#
#    plot_df = plots_list[file]
#    base_name = Path(file).stem
#
#    for _, row_index in source_df.groupby(q_bucket, sort=False).groups.items():
#        plot_df_split = plot_df.iloc[row_index].reset_index(drop=True)
#
#        qmin_vals = source_df.loc[row_index, "Q_min"].unique()
#        qmax_vals = source_df.loc[row_index, "Q_max"].unique()
#
#        if len(qmin_vals) == 1 and len(qmax_vals) == 1:
#            qmin, qmax = qmin_vals[0], qmax_vals[0]
#        else:
#
#            qmin = source_df.loc[row_index, "Q_min"].min()
#            qmax = source_df.loc[row_index, "Q_max"].max()
#
#        new_key = f"{base_name}_{fmt(qmin)}Q{fmt(qmax)}"
#        plots_list[new_key] = plot_df_split
#
#    del plots_list[file]

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pathlib import Path

# Order to plot (or: sorted(file_names))
files_in_order = list(plots_list.keys())
max_cols_per_row = 3

n_files = len(files_in_order)
if n_files == 0:
    raise ValueError("No files to plot.")

ncols = min(max_cols_per_row, n_files)
nrow_pairs = math.ceil(n_files / ncols)

# Figure & outer grid (spacing BETWEEN files)
fig = plt.figure(figsize=(4.6 * ncols, 3.8 * nrow_pairs))
outer = fig.add_gridspec(nrows=nrow_pairs, ncols=ncols, wspace=0.42, hspace=0.32)

# Hide only the "0" tick label on the spectrum axis
hide_zero_label = FuncFormatter(lambda y, pos: "" if np.isclose(y, 0.0) else f"{y:g}")

def set_spectrum_ylim(ax, y_data, y_err, y_pred, pad_frac=0.12):
    """
    Bottom fixed at 0. Top = max(data+err, prediction) * (1 + pad_frac).
    Purely percentage-based (no fixed absolute padding).
    """
    y_data = np.asarray(y_data, float)
    y_err  = np.asarray(y_err,  float)
    y_pred = np.asarray(y_pred, float)

    data_up = y_data + np.nan_to_num(y_err, nan=0.0)
    candidates = np.concatenate([
        data_up[np.isfinite(data_up)],
        y_pred[np.isfinite(y_pred)]
    ])
    if candidates.size == 0:
        ax.set_ylim(0.0, 1.0)
        return

    ymax = float(np.max(candidates))
    if not np.isfinite(ymax) or ymax <= 0:
        ax.set_ylim(0.0, 1.0)
        return

    ax.set_ylim(0.0, np.nextafter(ymax * (1.0 + pad_frac), np.inf))

def set_ratio_ylim_symmetric_around_one(ax, ratio_vals, ratio_errs,
                                        margin_frac=0.22, min_half_frac=0.08):
    """
    Symmetric about y=1.0, covering ratio points AND 1±err, with percentage-based extra margin.
      - margin_frac: extra headroom as a fraction of the required half-span
      - min_half_frac: minimum half-span as a fraction of 1.0
    """
    r = np.asarray(ratio_vals, float)
    e = np.asarray(ratio_errs, float)
    r_finite = r[np.isfinite(r)]
    e_finite = e[np.isfinite(e)]

    if r_finite.size == 0 and e_finite.size == 0:
        half = max(min_half_frac, 0.10)
        ax.set_ylim(1.0 - half, 1.0 + half)
        return

    lower_from_points = np.min(r_finite) if r_finite.size else np.inf
    upper_from_points = np.max(r_finite) if r_finite.size else -np.inf

    if e_finite.size:
        max_err = float(np.nanmax(e_finite))
        lower_from_errors = 1.0 - max_err
        upper_from_errors = 1.0 + max_err
    else:
        lower_from_errors = np.inf
        upper_from_errors = -np.inf

    lower = min(lower_from_points, lower_from_errors)
    upper = max(upper_from_points, upper_from_errors)

    half_needed = max(1.0 - lower, upper - 1.0, 0.0)
    half = max(half_needed * (1.0 + margin_frac), min_half_frac)
    ax.set_ylim(1.0 - half, 1.0 + half)

for idx, fname in enumerate(files_in_order):
    r = idx // ncols
    c = idx % ncols

    # Inner grid per file: top+bottom, zero vertical gap; share x within the pair
    inner = outer[r, c].subgridspec(nrows=2, ncols=1, height_ratios=[3, 1], hspace=0.0)
    ax_top = fig.add_subplot(inner[0, 0])
    ax_bot = fig.add_subplot(inner[1, 0], sharex=ax_top)

    plot_df = plots_list[fname]
    qT         = plot_df["qT_array"].to_numpy()
    data_vals  = plot_df["data_array"].to_numpy()
    pred_vals  = np.asarray(plot_df["prediction_array"], float)
    data_err   = plot_df["error_uncor_array"].to_numpy()
    ratio_vals = plot_df["ratio_array"].to_numpy()
    ratio_errs = plot_df["error_uncor_ratio_array"].to_numpy()

    # --- Top: data ± error vs prediction ---
    ax_top.errorbar(qT, data_vals, yerr=data_err, fmt='o', ms=3, elinewidth=1, capsize=2, label="data")
    ax_top.plot(qT, pred_vals, linewidth=1.5, label="prediction")
    ax_top.set_title(Path(fname).stem)
    ax_top.set_ylabel("dσ/dqT")
    ax_top.grid(True, alpha=0.3)
    set_spectrum_ylim(ax_top, data_vals, data_err, pred_vals)     # starts at 0, % headroom
    ax_top.yaxis.set_major_formatter(hide_zero_label)             # hide "0" label
    ax_top.tick_params(labelbottom=False)                         # no x labels on top panel
    # show the separator line between panels (keep both spines visible)
    ax_top.spines['bottom'].set_visible(True)
    #if idx == 0:
    #    ax_top.legend(frameon=False)

    # --- Bottom: ratio, symmetric about 1.0 with % margin ---
    ax_bot.axhline(1.0, linestyle="--", linewidth=1)
    ax_bot.plot(qT, ratio_vals, marker='o', linestyle='none', ms=3)
    ax_bot.errorbar(qT, np.ones_like(qT), yerr=ratio_errs, fmt='none', elinewidth=1, capsize=2)
    ax_bot.set_xlabel("qT [GeV]")
    ax_bot.set_ylabel("pred/data")
    ax_bot.grid(True, alpha=0.3)
    set_ratio_ylim_symmetric_around_one(ax_bot, ratio_vals, ratio_errs,
                                        margin_frac=0.22, min_half_frac=0.08)
    ax_bot.spines['top'].set_visible(True)  # separator remains visible

plt.tight_layout()
plt.show()